<a href="https://colab.research.google.com/github/arthurmraesdsilva-lgtm/codespaces-express/blob/main/Seguran%C3%A7a_em_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask PyJWT

In [ ]:
# Importa as bibliotecas necessárias
from flask import Flask, jsonify, request, make_response
import jwt
import datetime
from functools import wraps

# Cria a aplicação Flask
app = Flask(__name__)

# Chave secreta usada para gerar e validar os tokens JWT
app.config['SECRET_KEY'] = 'minha_chave_secreta'


# Função que verifica se o usuário enviou um token válido
def token_requerido(f):

    # Mantém as informações da função original
    @wraps(f)
    def decorator(*args, **kwargs):

        # Obtém o token enviado no cabeçalho da requisição
        token = request.headers.get('x-access-token')

        # Verifica se o token foi enviado
        if not token:
            return jsonify({
                'mensagem': 'Token de acesso é necessário!'
            }), 401

        try:
            # Decodifica e valida o token
            dados = jwt.decode(
                token,
                app.config['SECRET_KEY'],
                algorithms=["HS256"]
            )

            # Salva os dados do usuário na requisição
            request.user_data = dados

        except:
            # Retorna erro caso o token seja inválido
            return jsonify({
                'mensagem': 'Token inválido!'
            }), 401

        # Executa a função protegida
        return f(*args, **kwargs)

    return decorator


# Rota para realizar login e gerar um token JWT
@app.route('/login', methods=['POST'])
def login():

    # Recebe os dados enviados pelo usuário
    dados = request.get_json()

    # Verifica se o usuário e senha estão corretos
    if dados['usuario'] == 'admin' and dados['senha'] == '1234':

        # Cria o token com os dados do usuário
        token = jwt.encode(
            {
                'usuario': dados['usuario'],

                # Define validade de 30 minutos
                'exp': datetime.datetime.utcnow() +
                       datetime.timedelta(minutes=30)
            },
            app.config['SECRET_KEY']
        )

        # Retorna o token gerado
        return jsonify({'token': token})

    # Retorna erro caso login esteja incorreto
    return make_response(
        'Usuário ou senha incorretos!',
        401
    )


# Rota protegida para listar produtos
@app.route('/produtos', methods=['GET'])
@token_requerido
def listar_produtos():

    # Lista simulada de produtos
    produtos = [
        {
            "id": 1,
            "nome": "Camiseta",
            "preco": 50.00
        },
        {
            "id": 2,
            "nome": "Tênis",
            "preco": 120.00
        }
    ]

    # Retorna os produtos em formato JSON
    return jsonify(produtos)


# Rota para adicionar produtos
# Apenas administradores podem acessar
@app.route('/produtos', methods=['POST'])
@token_requerido
def adicionar_produto():

    # Verifica se o usuário é administrador
    if request.user_data['usuario'] != 'admin':
        return jsonify({
            'mensagem': 'Ação não permitida!'
        }), 403

    # Recebe os dados do novo produto
    novo_produto = request.get_json()

    # Aqui ficaria a lógica para salvar o produto
    # Exemplo: banco de dados ou lista

    return jsonify({
        'mensagem': 'Produto adicionado com sucesso!'
    }), 201


# Rota para remover um produto
# Apenas administradores podem acessar
@app.route('/produtos/<int:id>', methods=['DELETE'])
@token_requerido
def remover_produto(id):

    # Verifica se o usuário é administrador
    if request.user_data['usuario'] != 'admin':
        return jsonify({
            'mensagem': 'Ação não permitida!'
        }), 403

    # Aqui ficaria a lógica para remover o produto pelo ID

    return jsonify({
        'mensagem': 'Produto removido com sucesso!'
    }), 200


# Inicia a aplicação
if __name__ == '__main__':

    # Executa o servidor Flask em modo de depuração
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
